# PenuX — Automated Cureus Submission
**Run each cell in order. The final cell opens the Cureus form and fills it automatically.**

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────
!pip install playwright -q
!playwright install chromium
# Fix missing shared libraries (TargetClosedError / libXcomposite fix)
!apt-get install -y -q \
  libxcomposite1 libxdamage1 libxfixes3 libxrandr2 libgbm1 \
  libnss3 libatk1.0-0 libatk-bridge2.0-0 libdrm2 \
  libxkbcommon0 libasound2 libxshmfence1 libpango-1.0-0 \
  libcairo2 libcups2 libdbus-1-3 libexpat1 libfontconfig1 \
  libglib2.0-0 libnspr4 libx11-6 libx11-xcb1 libxcb1 \
  libxext6 libxrender1 libxtst6
print('✅ Ready')

In [ ]:
# ── Cell 2: Credentials ────────────────────────────────────────────────────
EMAIL    = 'nsh531@gmail.com'
PASSWORD = '318962420'
ORCID    = '0000-0002-3482-5508'
print(f'Account: {EMAIL}')

In [ ]:
# ── Cell 3: Manuscript content ─────────────────────────────────────────────
TITLE = (
    'PenuX: A Comparative Study of 11 Machine Learning and Deep Learning Models '
    'for Early Severity Prediction of Acute Pancreatitis Using Routine Admission '
    'Laboratory Values, with FHIR R4 Integration'
)

ABSTRACT = """Background
Severe Acute Pancreatitis (SAP) carries a mortality rate of 20-30% and requires early risk stratification. Classical scoring systems (Ranson, BISAP, APACHE II) require 24-48 hours of serial laboratory observation and lack electronic health record (EHR) integration.

Methods
Retrospective analysis of 722 acute pancreatitis (AP) admissions (585 severe / 137 mild; Atlanta 2012 classification) from a single Chinese tertiary institution. Eleven models were trained on 106 routine admission laboratory features using 5-fold stratified cross-validation: three classical ML models (Logistic Regression, Random Forest, Gradient Boosting), three MLP deep learning models, and five LSTM-based sequence models (Vanilla LSTM, Stacked LSTM, Bidirectional LSTM, LSTM+Attention, CNN-LSTM).

Results
Random Forest achieved AUC=0.877, F1=0.917, sensitivity=96.8%, specificity=38.7% at threshold 0.535. Gradient Boosting was comparable (AUC=0.874, sensitivity=97.1%). CNN-LSTM performed best among recurrent architectures (AUC=0.772, sensitivity=98.6%). Key predictors: calcium, D-dimer, LDH, lactate, hematocrit. A label inversion effect was identified: mild biliary AP cases showed higher WBC, CRP, and lipase than severe necrotising AP cases.

Conclusions
Random Forest achieves SAP triage with AUC=0.877 from a single admission blood draw, eliminating the 24-48 hour observation window required by classical scoring systems. The open-source PenuX platform provides FHIR R4, HL7 v2.x, and Israeli HIS (Camelion) integration. External validation is required before clinical use."""

KEYWORDS = [
    'acute pancreatitis', 'severe acute pancreatitis', 'machine learning',
    'deep learning', 'random forest', 'LSTM', 'FHIR R4', 'clinical prediction model'
]

print('✅ Content loaded —', len(ABSTRACT.split()), 'words in abstract')

In [ ]:
# ── Cell 4: Try sign-in (account may already exist) or sign-up ─────────────
import asyncio
from playwright.async_api import async_playwright
from IPython.display import Image, display
import base64, time

async def screenshot(page, path):
    await page.screenshot(path=path, full_page=True)
    display(Image(path))

async def run():
    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=True,
            args=['--no-sandbox', '--disable-dev-shm-usage']
        )
        ctx = await browser.new_context(
            ignore_https_errors=True,
            user_agent='Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36'
        )
        page = await ctx.new_page()

        # ── Try sign in first ──────────────────────────────────────────────
        print('Step 1: Attempting sign-in...')
        await page.goto('https://www.cureus.com/sign_in', wait_until='networkidle', timeout=30000)
        await screenshot(page, '/tmp/step1_signin.png')

        try:
            await page.fill('input[name="user[email]"]', EMAIL, timeout=5000)
            await page.fill('input[name="user[password]"]', PASSWORD, timeout=5000)
            await page.click('input[type="submit"]')
            await page.wait_for_load_state('networkidle', timeout=15000)
            await screenshot(page, '/tmp/step1b_after_signin.png')
            url = page.url
            print(f'After sign-in URL: {url}')

            if 'dashboard' in url or 'profile' in url:
                print('✅ Signed in to existing account!')
                return page, browser
            else:
                print('Sign-in failed — will try sign-up...')
        except Exception as e:
            print(f'Sign-in error: {e}')

        # ── Sign up ────────────────────────────────────────────────────────
        print('Step 2: Creating new account...')
        await page.goto('https://www.cureus.com/sign_up', wait_until='networkidle', timeout=30000)
        await screenshot(page, '/tmp/step2_signup.png')

        try:
            await page.fill('input[name="user[first_name]"]', 'Netanel', timeout=5000)
            await page.fill('input[name="user[last_name]"]', 'Shoshany', timeout=5000)
            await page.fill('input[name="user[email]"]', EMAIL, timeout=5000)
            await page.fill('input[name="user[password]"]', PASSWORD, timeout=5000)
            await page.fill('input[name="user[password_confirmation]"]', PASSWORD, timeout=5000)
            await screenshot(page, '/tmp/step2b_filled.png')
            await page.click('input[type="submit"]')
            await page.wait_for_load_state('networkidle', timeout=15000)
            await screenshot(page, '/tmp/step2c_after_signup.png')
            print(f'After sign-up URL: {page.url}')
        except Exception as e:
            print(f'Sign-up error: {e}')
            await screenshot(page, '/tmp/step2_error.png')

        await browser.close()

await run()

In [ ]:
# ── Cell 5: Check Gmail for verification email ─────────────────────────────
# If sign-up required email verification, run this cell after clicking the
# link in your inbox, then re-run Cell 4 to sign in.
print('If you received a verification email from Cureus:')
print('1. Open nsh531@gmail.com')
print('2. Click the confirmation link')
print('3. Then re-run Cell 4 — it will sign you in')
print()
print('Once signed in, continue to Cell 6 to submit the manuscript.')

In [ ]:
# ── Cell 6: Submit manuscript ──────────────────────────────────────────────
async def submit():
    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=True,
            args=['--no-sandbox', '--disable-dev-shm-usage']
        )
        ctx = await browser.new_context(
            ignore_https_errors=True,
            user_agent='Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36'
        )
        page = await ctx.new_page()

        # Sign in
        print('Signing in...')
        await page.goto('https://www.cureus.com/sign_in', wait_until='networkidle', timeout=30000)
        await page.fill('input[name="user[email]"]', EMAIL)
        await page.fill('input[name="user[password]"]', PASSWORD)
        await page.click('input[type="submit"]')
        await page.wait_for_load_state('networkidle', timeout=15000)
        print(f'Signed in. URL: {page.url}')

        # Navigate to submit
        print('Going to submission page...')
        await page.goto('https://www.cureus.com/submit', wait_until='networkidle', timeout=30000)
        await screenshot(page, '/tmp/submit1.png')

        # Click "Submit New Article"
        try:
            await page.click('text=Submit New Article', timeout=8000)
            await page.wait_for_load_state('networkidle')
        except:
            print('No Submit New Article button found — checking current form...')
        await screenshot(page, '/tmp/submit2_form.png')

        # Fill article type
        try:
            await page.select_option('select#article_type', label='Original Article', timeout=5000)
            print('✅ Article type: Original Article')
        except:
            print('⚠️  Could not select article type')

        # Fill title
        try:
            await page.fill('#article_title, input[name="article[title]"]', TITLE, timeout=5000)
            print('✅ Title filled')
        except Exception as e:
            print(f'⚠️  Title: {e}')

        # Fill abstract
        try:
            await page.fill('#article_abstract, textarea[name="article[abstract]"]', ABSTRACT, timeout=5000)
            print('✅ Abstract filled')
        except Exception as e:
            print(f'⚠️  Abstract: {e}')

        await screenshot(page, '/tmp/submit3_filled.png')
        print('\n📸 Screenshots saved: /tmp/submit1.png, submit2_form.png, submit3_filled.png')
        print('\n⚠️  REVIEW the screenshots above before proceeding!')
        print('Run Cell 7 to submit, or stop here to review manually.')
        await browser.close()

await submit()

In [ ]:
# ── Cell 7: FINAL SUBMIT (run only after reviewing screenshots) ────────────
CONFIRMED = True  # Set to True to confirm submission

async def final_submit():
    if not CONFIRMED:
        print('Set CONFIRMED = True to proceed.')
        return

    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=True,
            args=['--no-sandbox', '--disable-dev-shm-usage']
        )
        ctx = await browser.new_context(ignore_https_errors=True)
        page = await ctx.new_page()

        await page.goto('https://www.cureus.com/sign_in', wait_until='networkidle', timeout=30000)
        await page.fill('input[name="user[email]"]', EMAIL)
        await page.fill('input[name="user[password]"]', PASSWORD)
        await page.click('input[type="submit"]')
        await page.wait_for_load_state('networkidle', timeout=15000)

        await page.goto('https://www.cureus.com/submit', wait_until='networkidle', timeout=30000)
        try:
            await page.click('text=Submit New Article', timeout=5000)
            await page.wait_for_load_state('networkidle')
        except: pass

        await page.select_option('select#article_type', label='Original Article')
        await page.fill('#article_title, input[name="article[title]"]', TITLE)
        await page.fill('#article_abstract, textarea[name="article[abstract]"]', ABSTRACT)

        # Submit
        try:
            await page.click('input[type="submit"][value*="Submit"], button:has-text("Submit")', timeout=8000)
            await page.wait_for_load_state('networkidle', timeout=30000)
            await screenshot(page, '/tmp/final_confirmation.png')
            print(f'\n✅ SUBMITTED! URL: {page.url}')
        except Exception as e:
            print(f'Submit click failed: {e}')
            await screenshot(page, '/tmp/final_error.png')

        await browser.close()

await final_submit()